# Scaling an httpbin Deployment

This notebook creates an independent httpbin Tool and Deployment, starts with `MinInstanceCount=0` to observe on-demand startup, then updates the Deployment to keep two instances warm while changing the instance ceiling and per-instance request-concurrency lease. It demonstrates configuration and observable requests; it is not a load test.

> Copy IDs manually from command output. The notebook contains no extraction or polling scripts. Replace `AGR_ROLE_ARN` with a CAM role ARN that lets AGR pull the target CCR image.

In [ ]:
%env AGR_REGION=ap-shanghai
%env AGR_DOMAIN=tencentags.com
%env AGR_ROLE_ARN=qcs::cam::uin/replace-me:roleName/replace-me
%env HTTPBIN_TOOL_NAME=httpbin-scaling-your-name
%env HTTPBIN_DEPLOYMENT_NAME=httpbin-scaling-your-name
!agr status

## 1. Create an independent Tool

Replace `your-name` in both names before creating the Tool.

In [ ]:
!agr tool create \
  --region "$AGR_REGION" \
  --tool-name "$HTTPBIN_TOOL_NAME" \
  --tool-type custom \
  --persistent \
  --role-arn "$AGR_ROLE_ARN" \
  --network-configuration '{"NetworkMode":"PUBLIC"}' \
  --custom-configuration '{"Image":"ccr.ccs.tencentyun.com/ags.dev/go-httpbin:v2.25.0","ImageRegistryType":"personal","Command":["/bin/go-httpbin"],"Args":["-host","0.0.0.0","-port","8080"],"Env":[{"Name":"EXCLUDE_HEADERS","Value":"X-Access-Token"}],"Ports":[{"Name":"http","Port":8080,"Protocol":"TCP"}],"Resources":{"CPU":"200m","Memory":"500Mi"},"Probe":{"HttpGet":{"Path":"/status/200","Port":8080,"Scheme":"HTTP"},"ReadyTimeoutMs":30000,"ProbeTimeoutMs":1000,"ProbePeriodMs":3000,"SuccessThreshold":1,"FailureThreshold":10}}' \
  --wait

## 2. Start with zero instances

Copy `ToolId`. The initial configuration permits zero active instances, scales to at most three, and lets each instance hold one request or connection lease at a time. The first request triggers on-demand startup.

In [ ]:
%env HTTPBIN_TOOL_ID=sdt-replace-me
!agr deployment create \
  --region "$AGR_REGION" \
  --deployment-name "$HTTPBIN_DEPLOYMENT_NAME" \
  --tool-id "$HTTPBIN_TOOL_ID" \
  --scaling-configuration '{"MinInstanceCount":0,"MaxInstanceCount":3,"MaxInstanceRequestConcurrency":1}' \
  --lifecycle-configuration '{"IdleTimeoutSeconds":60,"IdleAction":"STOP"}'

## 3. Trigger on-demand startup

Copy `DeploymentId`, acquire a short-lived token, and then copy `Data.Response.Response.Token`. The first call may include instance startup latency; later calls normally reuse active capacity.

In [ ]:
%env HTTPBIN_DEPLOYMENT_ID=dpl-replace-me
!agr api call AcquireDeploymentToken --region "$AGR_REGION" --request '{"DeploymentId":"'$HTTPBIN_DEPLOYMENT_ID'"}' --output json

In [ ]:
%env HTTPBIN_DEPLOYMENT_TOKEN=dpt-replace-me
!curl --fail-with-body --silent --show-error \
  --header "X-Access-Token: $HTTPBIN_DEPLOYMENT_TOKEN" \
  "https://8080-$HTTPBIN_DEPLOYMENT_ID.$AGR_REGION.agents.$AGR_DOMAIN/get"
!agr deployment get "$HTTPBIN_DEPLOYMENT_ID" --region "$AGR_REGION"

## 4. Switch to warm capacity

`deployment update` replaces the scaling object in full, so all three fields must be present. The command below raises the active-instance floor to `2`, raises the ceiling to `4`, and lets each instance hold `10` request or connection leases. Use `get` to confirm the configuration; instance capacity converges asynchronously.

In [ ]:
!agr deployment update "$HTTPBIN_DEPLOYMENT_ID" \
  --region "$AGR_REGION" \
  --scaling-configuration '{"MinInstanceCount":2,"MaxInstanceCount":4,"MaxInstanceRequestConcurrency":10}'
!agr deployment get "$HTTPBIN_DEPLOYMENT_ID" --region "$AGR_REGION"

## Field meanings

- `MinInstanceCount` is the active-instance floor. Set it to `0` for scale-to-zero or to a positive value for warm capacity.
- `MaxInstanceCount` is the active-instance ceiling and cannot be lower than the floor.
- `MaxInstanceRequestConcurrency` limits simultaneous Deployment request or connection leases per active instance; it is not a global concurrency limit for the entire Deployment.

## 5. Clean up

If the observed behavior differs from expectations, retain the command output for diagnosis, then run cleanup.

In [ ]:
!agr deployment delete "$HTTPBIN_DEPLOYMENT_ID" --region "$AGR_REGION"
!agr instance list --tool-id "$HTTPBIN_TOOL_ID" --region "$AGR_REGION"

If any instance is not `STOPPED`, copy its ID. In a new cell, run `%env HTTPBIN_INSTANCE_ID=replace-me`, followed by `!agr instance delete "$HTTPBIN_INSTANCE_ID" --region "$AGR_REGION" --yes --wait`. Repeat as needed, then delete the Tool.

In [ ]:
!agr tool delete "$HTTPBIN_TOOL_ID" --region "$AGR_REGION" --yes --wait